In [1]:
import pandas as pd
import sqlite3

In [2]:
file_path = "../Datasets/retail_sales_raw.csv.csv"

df = pd.read_csv(file_path)

In [3]:
import os

print(os.getcwd())

C:\Users\vishw\OneDrive\Desktop\project_1\Notebooks


In [4]:
print(os.listdir(".."))


['dashboards', 'Datasets', 'images', 'Notebooks', 'notes', 'retail_sales.db']


In [5]:
print(os.listdir("../Datasets"))

['retail_sales_cleaned.csv', 'retail_sales_raw.csv.csv']


In [6]:
df.head()

,STATISTIC,Statistic Label,TLIST(A1),Year,C02583V03135,NACE Group,UNIT,VALUE
0,RSA08C01,Retail Sales Index Value Unadjusted,2021,2021,45,Motor trades (45),Base Year 2021=100,100.0
1,RSA08C01,Retail Sales Index Value Unadjusted,2021,2021,4711,Retail sale in non-specialised stores with foo...,Base Year 2021=100,100.0
2,RSA08C01,Retail Sales Index Value Unadjusted,2021,2021,4719,Department stores (4719),Base Year 2021=100,100.0
3,RSA08C01,Retail Sales Index Value Unadjusted,2021,2021,4730,Retail sale of automotive fuel (4730),Base Year 2021=100,100.0
4,RSA08C01,Retail Sales Index Value Unadjusted,2021,2021,4752,"Retail sale of hardware, paints and glass (4752)",Base Year 2021=100,100.0


In [7]:
df.shape


(440, 8)

In [8]:
df.columns.tolist()


['STATISTIC',
 'Statistic Label',
 'TLIST(A1)',
 'Year',
 'C02583V03135',
 'NACE Group',
 'UNIT',
 'VALUE']

In [9]:
df = df.rename(columns={
    'STATISTIC': 'statistic_code',
    'Statistic Label': 'statistic_label',
    'TLIST(A1)': 'time_code',
    'Year': 'year',
    'C02583V03135': 'nace_code',
    'NACE Group': 'retail_sector',
    'UNIT': 'unit',
    'VALUE': 'index_value'
})

In [10]:
df.columns.tolist()


['statistic_code',
 'statistic_label',
 'time_code',
 'year',
 'nace_code',
 'retail_sector',
 'unit',
 'index_value']

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   statistic_code   440 non-null    object 
 1   statistic_label  440 non-null    object 
 2   time_code        440 non-null    int64  
 3   year             440 non-null    int64  
 4   nace_code        440 non-null    object 
 5   retail_sector    440 non-null    object 
 6   unit             440 non-null    object 
 7   index_value      220 non-null    float64
dtypes: float64(1), int64(2), object(5)
memory usage: 27.6+ KB


In [12]:
missing_by_statistic = (
    df.groupby("statistic_label")["index_value"]
      .agg(
          total_rows="size",
          available_values="count"
      )
)
missing_by_statistic["missing_values"] = (
    missing_by_statistic["total_rows"]
    - missing_by_statistic["available_values"]
)

display(missing_by_statistic)

,total_rows,available_values,missing_values
statistic_label,,,
Retail Sales Index Value Adjusted,110,0,110
Retail Sales Index Value Unadjusted,110,110,0
Retail Sales Index Volume Adjusted,110,0,110
Retail Sales Index Volume Unadjusted,110,110,0


In [13]:
df_clean = df[df["index_value"].notna()].copy()

df_clean.shape

(220, 8)

In [14]:
df_clean.isna().sum()

statistic_code     0
statistic_label    0
time_code          0
year               0
nace_code          0
retail_sector      0
unit               0
index_value        0
dtype: int64

In [15]:
df_clean.duplicated().sum()

np.int64(0)

In [16]:
df_clean["statistic_label"].value_counts()

statistic_label
Retail Sales Index Value Unadjusted     110
Retail Sales Index Volume Unadjusted    110
Name: count, dtype: int64

In [17]:
df_clean["year"].unique()

array([2021, 2022, 2023, 2024, 2025])

In [18]:
df_clean["retail_sector"].nunique()

22

In [19]:
df_clean["retail_sector"].unique()

array(['Motor trades (45)',
       'Retail sale in non-specialised stores with food, beverages or tobacco predominating (4711)',
       'Department stores (4719)',
       'Retail sale of automotive fuel (4730)',
       'Retail sale of hardware, paints and glass (4752)',
       'Retail sale of furniture and lighting (4759)', 'Bars (5630)',
       'All retail businesses',
       'Motor trades and automotive fuel (45,4730)',
       'All retail businesses, excluding motor trades and bars',
       'All retail businesses, excluding motor trades',
       'All retail businesses, excluding motor trades, automotive fuel and bars',
       'Retail sale of food (4711,4721 to 4729)',
       'Retail sale of non food products, excluding motor trades, automotive fuel and bars',
       'Retail sale of food, beverages and tobacco in specialised stores (4721 to 4729)',
       'Retail sale of household equipment (4741 to 4743,4752,4754,4759)',
       'Retail sale of electrical goods (4741 to 4743,4754)',
 

In [20]:
df_clean["index_value"].describe()

count    220.000000
mean     114.386364
std       21.729255
min       82.200000
25%      100.000000
50%      110.100000
75%      122.675000
max      225.000000
Name: index_value, dtype: float64

In [21]:
output_path = "../Datasets/retail_sales_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully")

Cleaned dataset saved successfully


In [22]:
import sqlite3

database_path = "../retail_sales.db"

connection = sqlite3.connect(database_path)

df_clean.to_sql(
    name="retail_sales",
    con=connection,
    if_exists="replace",
    index=False
)

print("SQLite database and retail_sales table created successfully")

SQLite database and retail_sales table created successfully


In [23]:
query = """
SELECT *
FROM retail_sales
LIMIT 5;
"""

result = pd.read_sql_query(query, connection)

result

,statistic_code,statistic_label,time_code,year,nace_code,retail_sector,unit,index_value
0,RSA08C01,Retail Sales Index Value Unadjusted,2021,2021,45,Motor trades (45),Base Year 2021=100,100.0
1,RSA08C01,Retail Sales Index Value Unadjusted,2021,2021,4711,Retail sale in non-specialised stores with foo...,Base Year 2021=100,100.0
2,RSA08C01,Retail Sales Index Value Unadjusted,2021,2021,4719,Department stores (4719),Base Year 2021=100,100.0
3,RSA08C01,Retail Sales Index Value Unadjusted,2021,2021,4730,Retail sale of automotive fuel (4730),Base Year 2021=100,100.0
4,RSA08C01,Retail Sales Index Value Unadjusted,2021,2021,4752,"Retail sale of hardware, paints and glass (4752)",Base Year 2021=100,100.0


In [24]:
query = """
SELECT
    statistic_label,
    COUNT(*) AS total_rows
FROM retail_sales
GROUP BY statistic_label;
"""

result = pd.read_sql_query(query, connection)

result

,statistic_label,total_rows
0,Retail Sales Index Value Unadjusted,110
1,Retail Sales Index Volume Unadjusted,110


In [25]:
query = """
SELECT DISTINCT year
FROM retail_sales
ORDER BY year;
"""

years_result = pd.read_sql_query(query, connection)

years_result

,year
0,2021
1,2022
2,2023
3,2024
4,2025


In [26]:
query = """
SELECT
    year,
    statistic_label,
    index_value
FROM retail_sales
WHERE retail_sector = 'All retail businesses'
ORDER BY statistic_label, year;
"""

overall_trend = pd.read_sql_query(query, connection)

overall_trend

,year,statistic_label,index_value
0,2021,Retail Sales Index Value Unadjusted,100.0
1,2022,Retail Sales Index Value Unadjusted,113.3
2,2023,Retail Sales Index Value Unadjusted,122.2
3,2024,Retail Sales Index Value Unadjusted,123.6
4,2025,Retail Sales Index Value Unadjusted,126.9
5,2021,Retail Sales Index Volume Unadjusted,100.0
6,2022,Retail Sales Index Volume Unadjusted,107.4
7,2023,Retail Sales Index Volume Unadjusted,112.2
8,2024,Retail Sales Index Volume Unadjusted,112.8
9,2025,Retail Sales Index Volume Unadjusted,115.0


In [27]:
query = """
SELECT
    year,
    statistic_label,
    index_value
FROM retail_sales
WHERE retail_sector = 'All retail businesses'
ORDER BY statistic_label, year;
"""

overall_trend = pd.read_sql_query(query, connection)

overall_trend

,year,statistic_label,index_value
0,2021,Retail Sales Index Value Unadjusted,100.0
1,2022,Retail Sales Index Value Unadjusted,113.3
2,2023,Retail Sales Index Value Unadjusted,122.2
3,2024,Retail Sales Index Value Unadjusted,123.6
4,2025,Retail Sales Index Value Unadjusted,126.9
5,2021,Retail Sales Index Volume Unadjusted,100.0
6,2022,Retail Sales Index Volume Unadjusted,107.4
7,2023,Retail Sales Index Volume Unadjusted,112.2
8,2024,Retail Sales Index Volume Unadjusted,112.8
9,2025,Retail Sales Index Volume Unadjusted,115.0


In [28]:
query = """
SELECT
    year,

    MAX(
        CASE
            WHEN statistic_label = 'Retail Sales Index Value Unadjusted'
            THEN index_value
        END
    ) AS value_index,

    MAX(
        CASE
            WHEN statistic_label = 'Retail Sales Index Volume Unadjusted'
            THEN index_value
        END
    ) AS volume_index

FROM retail_sales

WHERE retail_sector = 'All retail businesses'

GROUP BY year

ORDER BY year;
"""

value_volume_trend = pd.read_sql_query(query, connection)

value_volume_trend

,year,value_index,volume_index
0,2021,100.0,100.0
1,2022,113.3,107.4
2,2023,122.2,112.2
3,2024,123.6,112.8
4,2025,126.9,115.0


In [29]:
query = """
WITH yearly_index AS (
    SELECT
        year,

        MAX(
            CASE
                WHEN statistic_label = 'Retail Sales Index Value Unadjusted'
                THEN index_value
            END
        ) AS value_index,

        MAX(
            CASE
                WHEN statistic_label = 'Retail Sales Index Volume Unadjusted'
                THEN index_value
            END
        ) AS volume_index

    FROM retail_sales

    WHERE retail_sector = 'All retail businesses'

    GROUP BY year
)

SELECT
    year,
    value_index,
    volume_index,
    ROUND(value_index - volume_index, 1) AS value_volume_gap

FROM yearly_index

ORDER BY year;
"""

gap_analysis = pd.read_sql_query(query, connection)

gap_analysis

,year,value_index,volume_index,value_volume_gap
0,2021,100.0,100.0,0.0
1,2022,113.3,107.4,5.9
2,2023,122.2,112.2,10.0
3,2024,123.6,112.8,10.8
4,2025,126.9,115.0,11.9


In [30]:
query = """
SELECT
    retail_sector,
    year,
    ROUND(index_value, 1) AS value_index
FROM retail_sales
WHERE year = (
    SELECT MAX(year)
    FROM retail_sales
)
  AND statistic_label = 'Retail Sales Index Value Unadjusted'
  AND nace_code NOT LIKE 'V%'
  AND nace_code NOT LIKE 'X%'
ORDER BY index_value DESC
LIMIT 5;
"""

top_value_sectors = pd.read_sql_query(query, connection)

top_value_sectors

,retail_sector,year,value_index
0,Bars (5630),2025,225.0
1,Motor trades (45),2025,132.5
2,Retail sale of automotive fuel (4730),2025,124.7
3,Retail sale of furniture and lighting (4759),2025,121.2
4,Department stores (4719),2025,120.5


In [31]:
query = """
SELECT
    retail_sector,
    year,
    ROUND(index_value, 1) AS value_index
FROM retail_sales
WHERE year = (
    SELECT MAX(year)
    FROM retail_sales
)
  AND statistic_label = 'Retail Sales Index Value Unadjusted'
  AND nace_code NOT LIKE 'V%'
  AND nace_code NOT LIKE 'X%'
ORDER BY index_value ASC
LIMIT 5;
"""

bottom_value_sectors = pd.read_sql_query(query, connection)

bottom_value_sectors

,retail_sector,year,value_index
0,"Retail sale of hardware, paints and glass (4752)",2025,106.6
1,Retail sale in non-specialised stores with foo...,2025,117.9
2,Department stores (4719),2025,120.5
3,Retail sale of furniture and lighting (4759),2025,121.2
4,Retail sale of automotive fuel (4730),2025,124.7


In [32]:
query = """
WITH sector_growth AS (
    SELECT
        retail_sector,

        MAX(
            CASE
                WHEN year = (
                    SELECT MIN(year)
                    FROM retail_sales
                )
                THEN index_value
            END
        ) AS first_year_index,

        MAX(
            CASE
                WHEN year = (
                    SELECT MAX(year)
                    FROM retail_sales
                )
                THEN index_value
            END
        ) AS latest_year_index

    FROM retail_sales

    WHERE statistic_label = 'Retail Sales Index Value Unadjusted'
      AND nace_code NOT LIKE 'V%'
      AND nace_code NOT LIKE 'X%'

    GROUP BY retail_sector
)

SELECT
    retail_sector,
    first_year_index,
    latest_year_index,
    ROUND(
        latest_year_index - first_year_index,
        1
    ) AS growth_index_points

FROM sector_growth

WHERE first_year_index IS NOT NULL
  AND latest_year_index IS NOT NULL

ORDER BY growth_index_points DESC;
"""

sector_growth_result = pd.read_sql_query(query, connection)

sector_growth_result

,retail_sector,first_year_index,latest_year_index,growth_index_points
0,Bars (5630),100.0,225.0,125.0
1,Motor trades (45),100.0,132.5,32.5
2,Retail sale of automotive fuel (4730),100.0,124.7,24.7
3,Retail sale of furniture and lighting (4759),100.0,121.2,21.2
4,Department stores (4719),100.0,120.5,20.5
5,Retail sale in non-specialised stores with foo...,100.0,117.9,17.9
6,"Retail sale of hardware, paints and glass (4752)",100.0,106.6,6.6


In [33]:
query = """
WITH sector_volume_growth AS (
    SELECT
        retail_sector,

        MAX(
            CASE
                WHEN year = (
                    SELECT MIN(year)
                    FROM retail_sales
                )
                THEN index_value
            END
        ) AS first_year_volume,

        MAX(
            CASE
                WHEN year = (
                    SELECT MAX(year)
                    FROM retail_sales
                )
                THEN index_value
            END
        ) AS latest_year_volume

    FROM retail_sales

    WHERE statistic_label = 'Retail Sales Index Volume Unadjusted'
      AND nace_code NOT LIKE 'V%'
      AND nace_code NOT LIKE 'X%'

    GROUP BY retail_sector
)

SELECT
    retail_sector,
    first_year_volume,
    latest_year_volume,
    ROUND(
        latest_year_volume - first_year_volume,
        1
    ) AS volume_growth_points

FROM sector_volume_growth

WHERE first_year_volume IS NOT NULL
  AND latest_year_volume IS NOT NULL

ORDER BY volume_growth_points DESC;
"""

sector_volume_growth_result = pd.read_sql_query(query, connection)

sector_volume_growth_result

,retail_sector,first_year_volume,latest_year_volume,volume_growth_points
0,Bars (5630),100.0,184.3,84.3
1,Retail sale of furniture and lighting (4759),100.0,122.9,22.9
2,Motor trades (45),100.0,116.9,16.9
3,Department stores (4719),100.0,112.6,12.6
4,Retail sale of automotive fuel (4730),100.0,104.6,4.6
5,"Retail sale of hardware, paints and glass (4752)",100.0,97.8,-2.2
6,Retail sale in non-specialised stores with foo...,100.0,97.1,-2.9


In [34]:
query = """
WITH sector_indexes AS (
    SELECT
        retail_sector,

        MAX(
            CASE
                WHEN year = (SELECT MIN(year) FROM retail_sales)
                 AND statistic_label = 'Retail Sales Index Value Unadjusted'
                THEN index_value
            END
        ) AS first_value_index,

        MAX(
            CASE
                WHEN year = (SELECT MAX(year) FROM retail_sales)
                 AND statistic_label = 'Retail Sales Index Value Unadjusted'
                THEN index_value
            END
        ) AS latest_value_index,

        MAX(
            CASE
                WHEN year = (SELECT MIN(year) FROM retail_sales)
                 AND statistic_label = 'Retail Sales Index Volume Unadjusted'
                THEN index_value
            END
        ) AS first_volume_index,

        MAX(
            CASE
                WHEN year = (SELECT MAX(year) FROM retail_sales)
                 AND statistic_label = 'Retail Sales Index Volume Unadjusted'
                THEN index_value
            END
        ) AS latest_volume_index

    FROM retail_sales

    WHERE nace_code NOT LIKE 'V%'
      AND nace_code NOT LIKE 'X%'

    GROUP BY retail_sector
)

SELECT
    retail_sector,

    ROUND(
        latest_value_index - first_value_index,
        1
    ) AS value_growth_points,

    ROUND(
        latest_volume_index - first_volume_index,
        1
    ) AS volume_growth_points,

    ROUND(
        (latest_value_index - first_value_index)
        -
        (latest_volume_index - first_volume_index),
        1
    ) AS growth_gap

FROM sector_indexes

WHERE first_value_index IS NOT NULL
  AND latest_value_index IS NOT NULL
  AND first_volume_index IS NOT NULL
  AND latest_volume_index IS NOT NULL

ORDER BY growth_gap DESC;
"""

value_volume_sector_comparison = pd.read_sql_query(query, connection)

value_volume_sector_comparison

,retail_sector,value_growth_points,volume_growth_points,growth_gap
0,Bars (5630),125.0,84.3,40.7
1,Retail sale in non-specialised stores with foo...,17.9,-2.9,20.8
2,Retail sale of automotive fuel (4730),24.7,4.6,20.1
3,Motor trades (45),32.5,16.9,15.6
4,"Retail sale of hardware, paints and glass (4752)",6.6,-2.2,8.8
5,Department stores (4719),20.5,12.6,7.9
6,Retail sale of furniture and lighting (4759),21.2,22.9,-1.7


In [36]:
excel_output_path = "../dashboards/retail_sales_analysis_results.xlsx"

with pd.ExcelWriter(excel_output_path, engine="openpyxl") as writer:
    overall_trend.to_excel(
        writer,
        sheet_name="Overall Trend",
        index=False
    )

    gap_analysis.to_excel(
        writer,
        sheet_name="Value Volume Gap",
        index=False
    )

    top_value_sectors.to_excel(
        writer,
        sheet_name="Top 5 Sectors",
        index=False
    )

    bottom_value_sectors.to_excel(
        writer,
        sheet_name="Bottom 5 Sectors",
        index=False
    )

    sector_growth_result.to_excel(
        writer,
        sheet_name="Value Growth",
        index=False
    )

    sector_volume_growth_result.to_excel(
        writer,
        sheet_name="Volume Growth",
        index=False
    )

    value_volume_sector_comparison.to_excel(
        writer,
        sheet_name="Growth Comparison",
        index=False
    )

print("SQL results exported successfully to Excel.")

PermissionError: [Errno 13] Permission denied: '../dashboards/retail_sales_analysis_results.xlsx'

In [37]:
value_volume_sector_comparison

,retail_sector,value_growth_points,volume_growth_points,growth_gap
0,Bars (5630),125.0,84.3,40.7
1,Retail sale in non-specialised stores with foo...,17.9,-2.9,20.8
2,Retail sale of automotive fuel (4730),24.7,4.6,20.1
3,Motor trades (45),32.5,16.9,15.6
4,"Retail sale of hardware, paints and glass (4752)",6.6,-2.2,8.8
5,Department stores (4719),20.5,12.6,7.9
6,Retail sale of furniture and lighting (4759),21.2,22.9,-1.7


In [38]:
excel_output_path = "../dashboards/retail_sales_analysis_results.xlsx"

with pd.ExcelWriter(
    excel_output_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    value_volume_sector_comparison.to_excel(
        writer,
        sheet_name="Growth Comparison",
        index=False
    )

print("Growth Comparison sheet updated successfully")

Growth Comparison sheet updated successfully
